# DEP Water Resource Matcher

Resolves `dont_know` and `ambiguous` planSource strings by fuzzy-matching them against
PA DEP water resource records (surface, ground, interconnection). DEP records carry
geolocated coordinates and an explicit withdrawal type that let us both classify
and geolocate sources that NHD matching could not handle.

**Input problem categories:**
- `dont_know` (267 unique sources, 10.9% of volume): SWW, SPWA, WI, Aqua, MAWC, NKWA, etc.
- `ambiguous` (763 unique sources, 4.2% of volume): no pattern matched at all

**DEP sources loaded:**
- `WaterResources2026_01.geojson` — all PA water resources (24,037 points, WGS84)
- Three shapefiles — surface / groundwater / interconnection pre-filtered by DEP type

In [1]:
import pandas as pd
import geopandas as gpd
import re
from rapidfuzz import fuzz, process
import warnings
warnings.filterwarnings('ignore')

DEP_GEOJSON   = 'data/PA resources/WaterResources2026_01.geojson'
DEP_SURFACE   = 'zip://data/PA resources/Water_Resources_-_Surface_Water_Withdrawal_Locations.zip'
DEP_GROUND    = 'zip://data/PA resources/Water_Resources_-_Ground_Water_Withdrawal_Locations.zip'
DEP_INTC      = 'zip://data/PA resources/Water_Resources_-_Interconnection_Locations.zip'
JUNCTION_PATH = 'data/junction_nhd_matched.parquet'
OUTPUT_PATH   = 'data/dep_match_results.parquet'

## 1. Load and combine DEP records

In [2]:
def load_shapefile(path, dep_type):
    gdf = gpd.read_file(path).to_crs('EPSG:4326')
    gdf['dep_type'] = dep_type
    gdf['lon'] = gdf.geometry.x
    gdf['lat'] = gdf.geometry.y
    return gdf[['ORGANIZATI', 'SUB_FACILI', 'SITE_STATU', 'dep_type', 'lat', 'lon']]

surf = load_shapefile(DEP_SURFACE, 'surface_water')
grnd = load_shapefile(DEP_GROUND,  'groundwater')
intc = load_shapefile(DEP_INTC,    'interconnection')

dep = pd.concat([surf, grnd, intc], ignore_index=True)
dep = dep[dep['SITE_STATU'].isin(['ACTIVE', 'INACTIVE'])]  # exclude abandoned/reclamation
dep = dep.dropna(subset=['lat', 'lon'])

print(f'DEP records loaded: {len(dep):,}')
print(dep['dep_type'].value_counts())

DEP records loaded: 14,717
dep_type
groundwater        8580
interconnection    3229
surface_water      2908
Name: count, dtype: int64


In [3]:
def normalize(s):
    """Lowercase, strip punctuation/noise tokens, collapse whitespace."""
    if pd.isna(s):
        return ''
    s = str(s).lower()
    s = re.sub(r'\bintc\b', '', s)
    s = re.sub(r'\bsource\s*\d+[a-z]?\b', '', s)
    s = re.sub(r'\bsww\b|\bwi\b', '', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def norm_operator(s):
    """Normalize operator name: remove LLC/INC/CO suffixes, collapse."""
    if pd.isna(s):
        return ''
    s = str(s).lower()
    s = re.sub(r'\b(llc|inc|corp|co|ltd|lp|llp|pllc|pc|ii|iii|iv)\b', '', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

# Strip type-prefix acronyms to isolate the location qualifier used for matching.
# e.g. 'Salsman SWW' -> 'Salsman', 'SPWA - College Ave' -> 'College Ave',
#      'Aqua Regional Pipeline System Docket #20120604' -> 'Regional Pipeline System'
_PREFIX_PAT = re.compile(
    r'^\s*(?:SPWA|SWPA|MAWC|NKWA|MANK|AWS)\s*[-–\s]*',
    re.IGNORECASE
)
_AQUA_PREFIX = re.compile(r'^\s*Aqua\s*[-–\s]*', re.IGNORECASE)
_SUFFIX_PAT  = re.compile(r'\s*\bSWW\b|\s*\bWI\b', re.IGNORECASE)
_DOCKET_PAT  = re.compile(r'Docket\s*#?\s*\d+', re.IGNORECASE)

def extract_key_name(s):
    """Strip acronym prefix/suffix to get the location qualifier for DEP matching."""
    s = _PREFIX_PAT.sub('', str(s))
    s = _AQUA_PREFIX.sub('', s)   # 'Aqua Regional Pipeline' -> 'Regional Pipeline'
    s = _SUFFIX_PAT.sub('', s)
    s = _DOCKET_PAT.sub('', s)
    return s.strip(' -–.')

dep['op_norm']  = dep['ORGANIZATI'].apply(norm_operator)
dep['src_norm'] = dep['SUB_FACILI'].apply(normalize)

dep.head(3)

,ORGANIZATI,SUB_FACILI,SITE_STATU,dep_type,lat,lon,op_norm,src_norm
0,BLEVINS FRUIT FARM,EBAUGHS CREEK,ACTIVE,surface_water,39.747680,-76.583278,blevins fruit farm,ebaughs creek
1,SWEPI LP,CROOKED CREEK,ACTIVE,surface_water,41.851667,-77.285278,swepi,crooked creek
2,MANHEIM TWP LANCASTER CNTY,IRRIGATION POND,ACTIVE,surface_water,40.080556,-76.324167,manheim twp lancaster cnty,irrigation pond


## 2. Load problem planSources

In [4]:
jct = pd.read_parquet(JUNCTION_PATH)
total_vol = jct['volume'].sum()

vol_by_source = (
    jct.groupby(['planSource', 'source_type', 'operator_clean'])
    ['volume'].sum()
    .reset_index()
)

# Problem pool: dont_know + ambiguous, deduplicated to unique planSource
prob = (
    vol_by_source[vol_by_source['source_type'].isin(['dont_know', 'ambiguous'])]
    .sort_values('volume', ascending=False)
    .drop_duplicates('planSource')
    .reset_index(drop=True)
)

prob['op_norm']  = prob['operator_clean'].apply(norm_operator)
prob['src_norm'] = prob['planSource'].apply(normalize)
# key_norm strips the type-prefix to leave the bare location qualifier
prob['key_norm'] = prob['planSource'].apply(lambda s: normalize(extract_key_name(s)))

print(f'Problem sources: {len(prob):,}')
print(f"  dont_know: {(prob['source_type']=='dont_know').sum()}")
print(f"  ambiguous: {(prob['source_type']=='ambiguous').sum()}")
print(f"Total volume: {prob['volume'].sum()/total_vol*100:.1f}% of all reported volume")
print()
print('Sample key_norm extractions:')
prob[['planSource','key_norm']].head(15)

Problem sources: 1,030
  dont_know: 267
  ambiguous: 763
Total volume: 14.4% of all reported volume

Sample key_norm extractions:


,planSource,key_norm
0,Salsman SWW,salsman
1,Newton SWW,newton
2,SPWA - Bambino,bambino
3,SPWA - Strosnider,strosnider
4,SPWA - Magnum,magnum
5,Monroe SWW,monroe
6,Aqua Regional Pipeline System Docket #20120604,regional pipeline system
7,Aqua Infrastructure Pipeline,infrastructure pipeline
8,Shaffer SWW,shaffer
9,SPWA - College Ave,college ave


## 3. Matching

**Scoring approach:** Two scores are computed and the max is taken:

- `token_set_ratio(key_norm, dep_src_norm)` — checks whether the bare location qualifier
  (e.g. "salsman", "college ave") is embedded anywhere in the DEP source name. Handles the
  case where the DEP name is much longer (e.g. "susquehanna river - salsman").
- `token_sort_ratio(src_norm, dep_src_norm)` — traditional full-name comparison; better
  for ambiguous sources where we have no type prefix to strip.

**Operator filter:** Candidates are first narrowed to DEP records whose `ORGANIZATI`
fuzzy-matches our `operator_clean` (score ≥ 70). If the best score from the operator-filtered
pool is < 70, a **global fallback** is also tried (necessary for utility-operated sources
like Aqua Infrastructure, SWPA Water Authority, which appear in DEP under the utility's
own name, not the fracking operator's). The best result across both passes is kept.

In [5]:
dep_ops = dep['op_norm'].unique()

def score_pool(key_norm, src_norm, pool):
    """Score a pool of DEP records against a query; return (best_score, best_row)."""
    if len(pool) == 0:
        return 0, None
    srcs = pool['src_norm'].tolist()
    # Score 1: key location qualifier embedded in DEP name
    r1 = process.extractOne(key_norm, srcs, scorer=fuzz.token_set_ratio)
    # Score 2: full normalized planSource vs DEP name
    r2 = process.extractOne(src_norm, srcs, scorer=fuzz.token_sort_ratio)
    best_score, best_idx = 0, 0
    for r in (r1, r2):
        if r and r[1] > best_score:
            best_score, best_idx = r[1], r[2]
    return best_score, pool.iloc[best_idx]

def get_op_pool(op_norm_query, dep_type_hint, op_thresh=70):
    """Return (operator-filtered pool, global pool), both optionally type-filtered."""
    if op_norm_query:
        matches = process.extract(
            op_norm_query, dep_ops,
            scorer=fuzz.token_set_ratio,
            limit=5,
            score_cutoff=op_thresh
        )
        if matches:
            op_pool = dep[dep['op_norm'].isin({m[0] for m in matches})].copy()
            op_pool['op_match'] = 'operator'
        else:
            op_pool = None
    else:
        op_pool = None

    global_pool = dep.copy()
    global_pool['op_match'] = 'global'

    def apply_type_hint(pool):
        if pool is None or dep_type_hint is None:
            return pool
        typed = pool[pool['dep_type'] == dep_type_hint]
        return typed if len(typed) > 0 else pool

    return apply_type_hint(op_pool), apply_type_hint(global_pool)

print('Scoring helpers built.')

Scoring helpers built.


In [6]:
# Pattern-based type hints for dont_know entries — these narrow the DEP search
def get_type_hint(plan_source):
    s = str(plan_source).upper()
    if re.search(r'\bSWW\b|\bWI\b', s):
        return 'surface_water'
    if re.search(r'\bSPWA\b|\bSWPA\b|\bAQUA\b|\bMAWC\b|\bNKWA\b|\bMANK\b|\bAWS\b', s):
        return 'interconnection'
    if re.search(r'\bBRINE\b', s):
        return None   # brine = reuse, won't be in DEP water resource records
    if re.search(r'QUARRY|LIMESTONE MINE', s):
        return 'groundwater'
    return None   # no hint — search all types

prob['dep_type_hint'] = prob['planSource'].apply(get_type_hint)
print(prob['dep_type_hint'].value_counts(dropna=False))

dep_type_hint
NaN                813
surface_water      131
interconnection     76
groundwater         10
Name: count, dtype: int64


In [7]:
_null_result = {'dep_org': None, 'dep_src': None, 'dep_type': None,
                'dep_lat': None, 'dep_lon': None, 'dep_score': 0, 'dep_op_match': None}

def match_one(row):
    key   = row['key_norm']
    src   = row['src_norm']
    hint  = row['dep_type_hint']
    op    = row['op_norm']

    op_pool, global_pool = get_op_pool(op, hint)

    # Pass 1: operator-filtered
    op_score, op_row = score_pool(key, src, op_pool) if op_pool is not None else (0, None)

    # Pass 2: global fallback if operator pass weak
    gl_score, gl_row = (0, None)
    if op_score < 70:
        gl_score, gl_row = score_pool(key, src, global_pool)

    if op_score >= gl_score and op_row is not None:
        best_score, best_row = op_score, op_row
    elif gl_row is not None:
        best_score, best_row = gl_score, gl_row
    else:
        return pd.Series(_null_result)

    return pd.Series({
        'dep_org':      best_row['ORGANIZATI'],
        'dep_src':      best_row['SUB_FACILI'],
        'dep_type':     best_row['dep_type'],
        'dep_lat':      best_row['lat'],
        'dep_lon':      best_row['lon'],
        'dep_score':    best_score,
        'dep_op_match': best_row['op_match'],
    })

print(f'Matching {len(prob)} problem sources against {len(dep):,} DEP records...')
match_cols = prob.apply(match_one, axis=1)
prob = pd.concat([prob, match_cols], axis=1)
print('Done.')

Matching 1030 problem sources against 14,717 DEP records...


Done.


## 4. Review results

In [8]:
prob['score_bin'] = pd.cut(
    prob['dep_score'],
    bins=[0, 59, 69, 79, 89, 100],
    labels=['<60', '60-69', '70-79', '80-89', '>=90']
)
print('DEP match score distribution:')
print(
    prob.groupby('score_bin', observed=True)['volume']
    .agg(sources='count', volume_Mgal=lambda x: x.sum() / 1e6)
    .round(1)
)

DEP match score distribution:
           sources  volume_Mgal
score_bin                      
<60            259       1484.0
60-69          152       1134.5
70-79          217       1369.8
80-89           96       1459.6
>=90           305       8921.6


In [9]:
display_cols = ['planSource', 'source_type', 'dep_score', 'dep_type',
                'dep_src', 'dep_org', 'dep_op_match', 'dep_lat', 'dep_lon',
                'volume']

print('=== High-confidence matches (score >= 80) ===')
high = prob[prob['dep_score'] >= 80].sort_values('volume', ascending=False)
print(f'{len(high)} sources  |  {high["volume"].sum()/1e6:,.0f} Mgal')
pd.set_option('display.max_colwidth', 60)
high[display_cols].head(40)

=== High-confidence matches (score >= 80) ===
398 sources  |  10,350 Mgal


,planSource,source_type,dep_score,dep_type,dep_src,dep_org,dep_op_match,dep_lat,dep_lon,volume
0,Salsman SWW,dont_know,100.000000,surface_water,SUSQUEHANNA RIVER - SALSMAN,CHESAPEAKE APPALACHIA LLC,operator,41.629914,-76.191763,9.279781e+08
1,Newton SWW,dont_know,100.000000,surface_water,SUSQUEHANNA RIVER (NEWTON),CHESAPEAKE APPALACHIA LLC,operator,41.690575,-76.276892,7.821915e+08
2,SPWA - Bambino,dont_know,100.000000,interconnection,SWPA BAMBINO VLT(EQUITRANS#19) SOURCE 141 INTC,EQT PROD CO,operator,39.887892,-80.481528,5.576238e+08
3,SPWA - Strosnider,dont_know,100.000000,interconnection,SWPA WATER AUTH STROSNIDER BULK WATER STN SOURCE 53 INTC,RICE DRILLING B LLC,operator,39.890000,-80.185556,5.005791e+08
4,SPWA - Magnum,dont_know,100.000000,interconnection,SWPA WATER AUTH MAGNUM MTR ON MAPLE RUN RD SOURCE 76 INTC,RICE DRILLING B LLC,operator,39.840967,-80.239300,4.191031e+08
5,Monroe SWW,dont_know,100.000000,surface_water,TOWANDA CREEK - MONROE HOSE,CHESAPEAKE APPALACHIA LLC,operator,41.721944,-76.465833,3.403902e+08
6,Aqua Regional Pipeline System Docket #20120604,dont_know,82.926829,interconnection,AQUA ETC REGIONAL PIPELINE SYS- AQUA INFRASTRUCTURE LLC ...,REPSOL OIL & GAS USA LLC,operator,41.486730,-77.174980,3.214495e+08
7,Aqua Infrastructure Pipeline,dont_know,100.000000,interconnection,AQUA ETC REGIONAL PIPELINE SYS- AQUA INFRASTRUCTURE LLC ...,REPSOL OIL & GAS USA LLC,global,41.486730,-77.174980,2.931616e+08
8,Shaffer SWW,dont_know,100.000000,surface_water,WYALUSING CREEK (SHAFFER),CHESAPEAKE APPALACHIA LLC,operator,41.676296,-76.251599,2.912217e+08
9,SPWA - College Ave,dont_know,100.000000,interconnection,SWPA WATER AUTH COLLEGE AVE MTR VLT SOURCE 95 INTC,EQT CHAP LLC,operator,39.897694,-79.880417,2.790906e+08


In [10]:
print('=== Mid-confidence matches (score 60-79) ===')
mid = prob[(prob['dep_score'] >= 60) & (prob['dep_score'] < 80)].sort_values('volume', ascending=False)
print(f'{len(mid)} sources  |  {mid["volume"].sum()/1e6:,.0f} Mgal')
mid[display_cols].head(40)

=== Mid-confidence matches (score 60-79) ===
368 sources  |  2,532 Mgal


,planSource,source_type,dep_score,dep_type,dep_src,dep_org,dep_op_match,dep_lat,dep_lon,volume
17,Aqua,dont_know,72.727273,interconnection,AQUA PA INTC,ROLLING GREEN GC,global,39.933333,-75.333333,151955881.0
22,MANK - Guyer Road,dont_know,66.666667,interconnection,NEW KENSINGTON CITY MUNI AUTH GUYER RD MTR VLT SOURCE 6 ...,"OLYMPUS ENERGY, LLC",operator,40.581300,-79.675000,127992371.0
25,Colwell SWW,dont_know,72.727273,surface_water,WELL,CHERRYS PKG INC,global,40.468333,-75.397778,117283084.0
27,Brine,dont_know,66.666667,groundwater,MINE,SILVERBROOK ANTHRACITE INC,global,41.290556,-75.807500,106574114.0
34,Ken Black TWL and Pump Pad,ambiguous,62.222222,groundwater,PUMP AND TREAT WELL,US SILICA CO,global,40.407666,-77.937850,91965447.0
35,MAWC - Snyder Road,dont_know,70.588235,interconnection,SNYDER BROS ALLEGHENY RIVER (COVE AREA) INTC,XTO ENERGY INC,operator,40.853806,-79.511444,91533876.0
36,SPWA - McNay Vantage,dont_know,70.000000,interconnection,VANTAGE ENERGY APPALACHIA II SOUTH FORK TENMILE CREEK 3 ...,VANTAGE ENERGY APPALACHIA LLC,global,39.910278,-80.112500,91496889.0
37,SWNPC Colwell Extraction,ambiguous,74.418605,groundwater,LTP EXTRACTION WELL,CHESTER CNTY SWA,global,40.111667,-75.955556,90135696.0
42,MAWC - Westmoreland County Blank Road,dont_know,73.076923,interconnection,WESTMORELAND CNTY AUTH INTC,MIEKA LLC,global,40.314917,-79.523250,80653022.0
45,MAWC - Blank Road,dont_know,66.666667,interconnection,MAWC SWEENEY PLT METER VLT @ BLANK RD SOURCE 10 INTC,"OLYMPUS ENERGY, LLC",operator,40.377611,-79.627000,74058188.0


In [11]:
print('=== Low-confidence / unmatched (score < 60) ===')
low = prob[prob['dep_score'] < 60].sort_values('volume', ascending=False)
print(f'{len(low)} sources  |  {low["volume"].sum()/1e6:,.0f} Mgal')
low[display_cols].head(40)

=== Low-confidence / unmatched (score < 60) ===
264 sources  |  1,488 Mgal


,planSource,source_type,dep_score,dep_type,dep_src,dep_org,dep_op_match,dep_lat,dep_lon,volume
13,SWN OPERATED SITES,ambiguous,58.823529,groundwater,PRESTRESS SPRING,NEW ENTERPRISE STONE & LIME CO INC,global,40.515611,-78.397583,2.030467e+08
31,MAWC,dont_know,42.857143,interconnection,MARION TWP INTC,RANGE RESOURCES APPALACHIA LLC,global,40.780778,-80.200944,9.486554e+07
32,Vargason WI,dont_know,55.555556,surface_water,ALMAS POND,Unavailable,global,40.865278,-76.228333,9.326382e+07
41,Parys WI,dont_know,50.000000,surface_water,SOURCE 9 MCCARTY,CHESAPEAKE APPALACHIA LLC,operator,41.652500,-76.258611,8.325428e+07
44,Hatch WI,dont_know,55.555556,surface_water,HATCHERY POND,HIDDEN VALLEY RESORT LTD PARTNERSHIP,global,40.056111,-79.255278,7.501421e+07
47,McKernan WI,dont_know,57.142857,surface_water,SHERMAN CREEK,TROUT BROS LLC,global,40.324333,-77.477444,7.137106e+07
58,Comtech,ambiguous,50.000000,surface_water,LAKE COMO,MT TONE SKI RESORT INC,global,41.851111,-75.347778,5.162085e+07
59,Forskville SWW,dont_know,54.545455,surface_water,VILLAGE POND,SEVEN SPRINGS MTN RESORT,global,40.017424,-79.302820,4.878088e+07
61,Mama Bear P1,ambiguous,57.142857,surface_water,BEAR CREEK,PA FISH & BOAT COMM FISHERIES BUR,global,42.035149,-80.220165,4.591944e+07
66,Lecrone WI,dont_know,58.823529,surface_water,LOWER POND,PINE CREST GC INC,global,40.262222,-75.256667,4.176745e+07


## 5. dont_know pattern breakdown

Review each major `dont_know` acronym group separately — these often have a clear expected
DEP type and the match quality tells us whether the DEP records cover them.

In [12]:
dk = prob[prob['source_type'] == 'dont_know'].copy()

pattern_groups = {
    'SWW':      r'\bSWW\b',
    'SPWA':     r'\bSPWA\b|\bSWPA\b',
    'WI':       r'\bWI\b',
    'Aqua':     r'\bAqua\b',
    'MAWC':     r'\bMAWC\b',
    'NKWA':     r'\bNKWA\b',
    'MANK':     r'\bMANK\b',
    'Clermont': r'\bClermont\b',
    'brine':    r'\bbrine\b',
    'quarry':   r'quarry',
}

for name, pat in pattern_groups.items():
    sub = dk[dk['planSource'].str.contains(pat, case=False, na=False)]
    if len(sub) == 0:
        continue
    good = (sub['dep_score'] >= 80).sum()
    total_n = len(sub)
    vol = sub['volume'].sum() / 1e6
    print(f"\n{name:10s}: {total_n:3d} sources  {vol:>10,.0f} Mgal  "
          f"score>=80: {good}/{total_n}")
    top = sub.sort_values('volume', ascending=False).head(6)
    for _, r in top.iterrows():
        print(f"  score={r['dep_score']:3.0f}  '{r['planSource'][:45]}'  ->  '{str(r['dep_src'])[:50]}'")


SWW       :  45 sources       3,865 Mgal  score>=80: 25/45
  score=100  'Salsman SWW'  ->  'SUSQUEHANNA RIVER - SALSMAN'
  score=100  'Newton SWW'  ->  'SUSQUEHANNA RIVER (NEWTON)'
  score=100  'Monroe SWW'  ->  'TOWANDA CREEK - MONROE HOSE'
  score=100  'Shaffer SWW'  ->  'WYALUSING CREEK (SHAFFER)'
  score=100  'Tiffany SWW'  ->  'SUSQUEHANNA RIVER (TIFFANY)'
  score=100  'SHAFFER SWW'  ->  'WYALUSING CREEK (SHAFFER)'

SPWA      :  35 sources       2,244 Mgal  score>=80: 26/35
  score=100  'SPWA - Bambino'  ->  'SWPA BAMBINO VLT(EQUITRANS#19) SOURCE 141 INTC'
  score=100  'SPWA - Strosnider'  ->  'SWPA WATER AUTH STROSNIDER BULK WATER STN SOURCE 5'
  score=100  'SPWA - Magnum'  ->  'SWPA WATER AUTH MAGNUM MTR ON MAPLE RUN RD SOURCE '
  score=100  'SPWA - College Ave'  ->  'SWPA WATER AUTH COLLEGE AVE MTR VLT SOURCE 95 INTC'
  score=100  'SWPA - Strosnider'  ->  'SWPA WATER AUTH STROSNIDER BULK WATER STN SOURCE 5'
  score= 70  'SPWA - McNay Vantage'  ->  'VANTAGE ENERGY APPALACHIA II

## 6. Save results

In [13]:
out = prob[[
    'planSource', 'source_type', 'operator_clean', 'volume',
    'dep_type_hint', 'dep_score', 'score_bin',
    'dep_org', 'dep_src', 'dep_type',
    'dep_lat', 'dep_lon', 'dep_op_match'
]].copy()

out.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved {len(out):,} rows to {OUTPUT_PATH}')

# Summary
good = out[out['dep_score'] >= 80]
mid  = out[(out['dep_score'] >= 60) & (out['dep_score'] < 80)]
low  = out[out['dep_score'] < 60]
print(f"\nHigh (>=80): {len(good):3d} sources  {good['volume'].sum()/1e6:>10,.0f} Mgal")
print(f"Mid  (60-79): {len(mid):3d} sources  {mid['volume'].sum()/1e6:>10,.0f} Mgal")
print(f"Low  (<60):  {len(low):3d} sources  {low['volume'].sum()/1e6:>10,.0f} Mgal")

Saved 1,030 rows to data/dep_match_results.parquet

High (>=80): 398 sources      10,350 Mgal
Mid  (60-79): 368 sources       2,532 Mgal
Low  (<60):  264 sources       1,488 Mgal


## 7. Next steps

After reviewing the matches above:

1. **High-confidence (≥80):** Accept DEP coordinates and DEP type classification. Update
   `source_type` in the junction table and backfill source coordinates.

2. **Mid-confidence (60–79):** Manual spot-check. Many will be correct; some may need
   the planSource string trimmed (operator prefix interference).

3. **Low-confidence / unmatched:** Investigate case-by-case:
   - `brine` entries → reclassify as `reuse` directly (no DEP record expected)
   - `Clermont`, `NKWA`, `MANK` → may need direct lookup by name in the broader DEP dataset
   - `ambiguous` residual → rule refinement in `source_classifier.ipynb`

4. **NHD re-match candidates:** Newly typed surface/impoundment sources with DEP coordinates
   can be fed back into `nhd_matcher.ipynb` with the precise DEP lat/lon, likely improving
   match scores for the 'fair' tier entries.

## 8. Full-sources DEP match (all source types)

Run the same DEP match across **all** planSources in the junction table — not just
`dont_know`/`ambiguous`. This serves two purposes:

- **Validation**: flag type conflicts where our classifier disagrees with DEP
- **Coordinates**: add DEP withdrawal-point lat/lon for every matched source, including
  603 `interconnection` sources that currently have no coordinates at all

Sources skipped: `reuse` and `no_source` (produced water / blank entries have no DEP record).

In [14]:
# Type hint for the full match: use declared source_type to pre-filter DEP pool,
# falling back to pattern-based hints for dont_know/ambiguous.
_STYPE_TO_DEP = {
    'surface_direct': 'surface_water',
    'impoundment':    'surface_water',
    'srbc_only':      'surface_water',
    'interconnection':'interconnection',
    'groundwater':    'groundwater',
}

def get_type_hint_full(row):
    stype = row['source_type']
    if stype in _STYPE_TO_DEP:
        return _STYPE_TO_DEP[stype]
    if stype in ('dont_know', 'ambiguous'):
        return get_type_hint(row['planSource'])   # pattern-based from earlier
    return None   # reuse, no_source — will be excluded below

# Build the full candidate table: one row per unique planSource, excluding reuse/no_source
SKIP_TYPES = {'reuse', 'no_source'}

all_sources = (
    jct.groupby(['planSource', 'source_type', 'operator_clean'])['volume']
    .sum()
    .reset_index()
    .sort_values('volume', ascending=False)
    .drop_duplicates('planSource')
    .reset_index(drop=True)
)
all_sources = all_sources[~all_sources['source_type'].isin(SKIP_TYPES)].copy()

all_sources['op_norm']       = all_sources['operator_clean'].apply(norm_operator)
all_sources['src_norm']      = all_sources['planSource'].apply(normalize)
all_sources['key_norm']      = all_sources['planSource'].apply(lambda s: normalize(extract_key_name(s)))
all_sources['dep_type_hint'] = all_sources.apply(get_type_hint_full, axis=1)

print(f'Full match candidate sources: {len(all_sources):,}')
print(all_sources.groupby('source_type').size().sort_values(ascending=False))
print(f"\nType hints assigned:")
print(all_sources['dep_type_hint'].value_counts(dropna=False))

Full match candidate sources: 3,506


source_type
surface_direct     1009
ambiguous           763
impoundment         684
interconnection     603
dont_know           267
groundwater         126
srbc_only            54
dtype: int64

Type hints assigned:
dep_type_hint
surface_water      1878
NaN                 813
interconnection     679
groundwater         136
Name: count, dtype: int64


In [15]:
print(f'Running full DEP match on {len(all_sources):,} sources against {len(dep):,} DEP records...')
full_match_cols = all_sources.apply(match_one, axis=1)
all_sources = pd.concat([all_sources, full_match_cols], axis=1)

all_sources['score_bin'] = pd.cut(
    all_sources['dep_score'],
    bins=[0, 59, 69, 79, 89, 100],
    labels=['<60', '60-69', '70-79', '80-89', '>=90']
)

print('Done.\n')
print('Score distribution by source_type:')
print(
    all_sources.groupby(['source_type', 'score_bin'], observed=True)
    .agg(sources=('planSource','count'), vol_Mgal=('volume', lambda x: x.sum()/1e6))
    .round(1)
    .to_string()
)

# Save full results
ALL_OUTPUT_PATH = 'data/dep_match_results_all.parquet'
all_sources[[
    'planSource', 'source_type', 'operator_clean', 'volume',
    'dep_type_hint', 'dep_score', 'score_bin',
    'dep_org', 'dep_src', 'dep_type',
    'dep_lat', 'dep_lon', 'dep_op_match'
]].to_parquet(ALL_OUTPUT_PATH, index=False)
print(f'\nSaved to {ALL_OUTPUT_PATH}')

Running full DEP match on 3,506 sources against 14,717 DEP records...


Done.

Score distribution by source_type:
                           sources  vol_Mgal
source_type     score_bin                   
ambiguous       <60            189     680.9
                60-69          111     535.3
                70-79          184     683.3
                80-89           82     654.3
                >=90           196    1441.1
dont_know       <60             70     803.1
                60-69           41     599.2
                70-79           33     686.5
                80-89           14     805.3
                >=90           109    7480.6
groundwater     70-79            2       1.0
                80-89            7      15.2
                >=90           117    1287.3
impoundment     <60              3       0.5
                60-69            2      69.5
                70-79            5     327.6
                80-89           54    2438.6
                >=90           620   12958.5
interconnection <60             10     281.4
             

## 9. Type conflict review

Compare what our classifier assigned vs. what DEP says for high-confidence matches (score ≥ 80).
Conflicts where DEP disagrees with our type warrant investigation.

In [16]:
# Map DEP type back to our source_type vocabulary for comparison
_DEP_TO_STYPE = {
    'surface_water':  'surface_direct',
    'interconnection':'interconnection',
    'groundwater':    'groundwater',
}

high_conf = all_sources[all_sources['dep_score'] >= 80].copy()
high_conf['dep_stype'] = high_conf['dep_type'].map(_DEP_TO_STYPE)

# A conflict is where our source_type doesn't match the DEP-implied type,
# excluding dont_know/ambiguous (those are expected to change)
expected_same = ~high_conf['source_type'].isin(['dont_know', 'ambiguous'])
conflict_mask = (
    expected_same &
    high_conf['dep_stype'].notna() &
    (high_conf['source_type'] != high_conf['dep_stype'])
)
conflicts = high_conf[conflict_mask].sort_values('volume', ascending=False)

print(f'High-confidence matches (score>=80): {len(high_conf):,}')
print(f'Type conflicts (our type != DEP type): {len(conflicts)}')

if len(conflicts):
    print()
    for _, r in conflicts.iterrows():
        print(f"  score={r['dep_score']:3.0f}  ours={r['source_type']:15s}  dep={r['dep_stype']:15s}  "
              f"vol={r['volume']/1e6:>8,.1f}Mgal  '{r['planSource'][:40]}'  ->  '{str(r['dep_src'])[:40]}'")

# Confusion matrix for all source types
print('\nDEP type vs our type (high-confidence matches, count):')
print(
    high_conf.groupby(['source_type','dep_stype'], observed=False)
    .size().unstack(fill_value=0)
    .to_string()
)

High-confidence matches (score>=80): 2,363
Type conflicts (our type != DEP type): 697

  score= 89  ours=impoundment      dep=surface_direct   vol= 1,370.1Mgal  'Cabot, Susquehanna #3 (Great Bend Impoun'  ->  'SUSQUEHANNA RIVER-3 (GREAT BEND)'
  score=100  ours=impoundment      dep=surface_direct   vol=   895.1Mgal  'YOUNG IMPOUNDMENT'  ->  'IMPOUNDMENT'
  score= 94  ours=impoundment      dep=surface_direct   vol=   840.4Mgal  'Cabot, Meshoppen Creek (Clark Impoundmen'  ->  'MESHOPPEN CREEK 1'
  score=100  ours=impoundment      dep=surface_direct   vol=   702.6Mgal  'Parys Water Impoundment'  ->  'IMPOUNDMENT'
  score=100  ours=impoundment      dep=surface_direct   vol=   549.6Mgal  'ZEFFER IMPOUNDMENT'  ->  'IMPOUNDMENT'
  score=100  ours=impoundment      dep=surface_direct   vol=   541.1Mgal  'BKV, Carrizo- E. B Wyalusing (Bonnice Im'  ->  'IMPOUNDMENT'
  score=100  ours=impoundment      dep=surface_direct   vol=   424.4Mgal  'Shaffer Water Impoundment'  ->  'IMPOUNDMENT'
  score= 94

## 10. Apply to junction table

Apply accepted DEP matches to `junction_nhd_matched.parquet`:

- **All sources, score ≥ 80**: add `dep_lat`, `dep_lon`, `dep_score`, `dep_src` columns
- **`dont_know` / `ambiguous`, score ≥ 80**: also update `source_type` to DEP-derived type
- **`brine` entries**: reclassify directly to `reuse` (no DEP record needed)

Output: `data/junction_dep_updated.parquet`

In [17]:
UPDATED_PATH = 'data/junction_dep_updated.parquet'

# Build the coordinate + type lookup from full match results
dep_lookup = all_sources[[
    'planSource', 'dep_score', 'dep_type', 'dep_lat', 'dep_lon', 'dep_src'
]].copy()

# Map DEP type to our source_type vocabulary
dep_lookup['dep_stype'] = dep_lookup['dep_type'].map(_DEP_TO_STYPE)

# Only carry forward coordinates for high-confidence matches
dep_lookup.loc[dep_lookup['dep_score'] < 80, ['dep_lat', 'dep_lon']] = None

# Start from the existing junction table
jct_out = jct.copy()

# Join DEP columns
jct_out = jct_out.merge(dep_lookup, on='planSource', how='left')

# --- Type reclassification ---
# 1. dont_know / ambiguous with DEP score >= 80: use DEP-derived type
reclassify_mask = (
    jct_out['source_type'].isin(['dont_know', 'ambiguous']) &
    (jct_out['dep_score'] >= 80) &
    jct_out['dep_stype'].notna()
)
jct_out.loc[reclassify_mask, 'source_type'] = jct_out.loc[reclassify_mask, 'dep_stype']

# 2. Brine → reuse unconditionally: check planSource content, not current type.
#    This overrides any DEP match (e.g. "Brine Water" scoring against "RAIN WATER").
brine_mask = jct_out['planSource'].str.contains(r'\bbrine\b', case=False, na=False)
jct_out.loc[brine_mask, 'source_type'] = 'reuse'

# --- Summary ---
print('=== Junction table after DEP reclassification ===')
orig_types = jct['source_type'].value_counts().rename('original')
new_types  = jct_out['source_type'].value_counts().rename('after_dep')
comparison = pd.concat([orig_types, new_types], axis=1).fillna(0).astype(int)
comparison['delta'] = comparison['after_dep'] - comparison['original']
print(comparison.sort_values('original', ascending=False).to_string())

print(f'\nRows with DEP coordinates: {jct_out["dep_lat"].notna().sum():,} '
      f'({jct_out["dep_lat"].notna().mean()*100:.1f}%)')

=== Junction table after DEP reclassification ===
                 original  after_dep  delta
source_type                                
surface_direct      15876      17154   1278
no_source            8551       8551      0
impoundment          8013       8013      0
interconnection      5193       6797   1604
ambiguous            4678       2845  -1833
dont_know            2954        814  -2140
groundwater          2053       2869    816
reuse                1570       1845    275
srbc_only             475        475      0

Rows with DEP coordinates: 28,198 (57.1%)


In [18]:
jct_out.to_parquet(UPDATED_PATH, index=False)
print(f'Saved {len(jct_out):,} rows to {UPDATED_PATH}')
print(f'Columns: {jct_out.columns.tolist()}')

Saved 49,363 rows to data/junction_dep_updated.parquet
Columns: ['api10', 'operator', 'planSource', 'managementPlanId', 'volume', 'saved_fn', 'operator_clean', 'site_clean', 'site_ID', 'source_type', 'search_name', 'nhd_id', 'nhd_name', 'nhd_ftype', 'nhd_layer', 'score', 'dist_km', 'match_tier', 'dep_score', 'dep_type', 'dep_lat', 'dep_lon', 'dep_src', 'dep_stype']


## 11. Rule-based fixes for residual dont_know / ambiguous

Apply targeted regex rules to the sources that survived DEP matching unresolved.
Rules are applied in priority order; first match wins within each remaining source.

Groups resolved here:
- **PA American / PAW** → `interconnection`
- **MAWC / MANK** → `interconnection`
- **Aqua variants** → `interconnection`
- **SPWA typo variants** (SPWMA, SPWRA) → `interconnection`
- **SGFS / service company names** → `interconnection`
- **WI** (private impoundments, owner name + WI) → `impoundment`
- **WTR IMP / FWI / misspelled impoundment** → `impoundment`
- **BRAD ## WTR IMP** (Bradford County ponds) → `impoundment`
- **SWN/SWNPC operated sites** → `impoundment`
- **Surface water names missed by regex** (Crk, Susq R, etc.) → `surface_direct`
- **SRBC # without "Docket" keyword** → `srbc_only`
- **Date-suffixed variants** → inherit type of base source name

In [19]:
# Rules applied only to rows still classified as dont_know or ambiguous.
# Priority order: first match wins.
RULE_FIXES = [
    # interconnection
    ('PA American / PAW',   r'PA\s+Amer|\bPAW\s*[-–]|Limbacher\s+Lane|Fox.s\s+Water\s+Limbacher',  'interconnection'),
    ('MAWC / MANK',         r'\bMAWC\b|\bMANK\b',                                                         'interconnection'),
    ('Aqua variants',       r'\bAqua\b|\bAQUA\b',                                                          'interconnection'),
    ('SPWA typo variants',  r'\bSPWMA\b|\bSPWRA\b|\bSPWA\b|\bSWPA\b',                                    'interconnection'),
    ('SGFS / service cos',  r'\bSGFS\b|Fluid\s+Recovery|Highland\s+Field\s+Serv|Independent\s+Water'
                            r'|Comtech|Laurel\s+Mountain\s+Prod|Blossburg',                                'interconnection'),
    # impoundment
    ('WI private impound',  r'\bWI\b',                                                                     'impoundment'),
    ('WTR IMP / FWI',       r'WTR\s+IMP|\bFWI\b|WATER\s+IMPOUND|IMPOUNDENT|IMPOUNDMNET',                 'impoundment'),
    ('BRAD ## WTR IMP',     r'\bBRAD\s+\d+',                                                              'impoundment'),
    ('SWN operated sites',  r'\bSWNPC?\b|\bSWN\s+(?:OPERATED|HINKLE|HINKIE|OPERATING)',                   'impoundment'),
    # surface_direct
    ('Surface water missed', r'Susq\s+R\b|\bCrk\b|\bBLACK\s+WALNUT\b|Ten\s+Mile.Cogar|Bowman\s+Crk',    'surface_direct'),
    # srbc_only
    ('SRBC # variant',      r'SRBC\s*#\s*\d+',                                                            'srbc_only'),
    # reuse
    ('Recycled/transfer',   r'Recylcled|Recycled|Freshwater\s+transfer\s+from',                           'reuse'),
]

# Build a lookup of already-resolved source → type for the date-suffix inheritance step.
# Use the final jct_out state after DEP reclassification.
resolved_types = (
    jct_out[~jct_out['source_type'].isin(['dont_know', 'ambiguous'])]
    .drop_duplicates('planSource')
    .set_index('planSource')['source_type']
)

_DATE_SUFFIX = re.compile(r'\s*[-_]?\s*\d{8}\s*$|\s*[-_]\s*\d{4}[-/]\d{2}[-/]\d{2}\s*$')

def apply_rule_fixes(df):
    """Apply rule-based fixes to rows still classified as dont_know or ambiguous."""
    df = df.copy()
    mask_resid = df['source_type'].isin(['dont_know', 'ambiguous'])

    # 1. Date-suffix inheritance: strip date, look up base source type
    date_mask = mask_resid & df['planSource'].str.contains(r'\d{8}', na=False)
    for idx in df[date_mask].index:
        base = _DATE_SUFFIX.sub('', df.at[idx, 'planSource']).strip()
        if base in resolved_types:
            df.at[idx, 'source_type'] = resolved_types[base]
            mask_resid[idx] = False

    # 2. Regex rules (first match wins)
    for name, pattern, new_type in RULE_FIXES:
        hit = mask_resid & df['planSource'].str.contains(pattern, case=False, na=False, regex=True)
        df.loc[hit, 'source_type'] = new_type
        mask_resid = mask_resid & ~hit   # clear matched rows

    return df, mask_resid

jct_out, still_resid = apply_rule_fixes(jct_out)

# Report
print('=== Rule-fix results ===')
fixed = ~jct_out['source_type'].isin(['dont_know', 'ambiguous']) & \
         jct['source_type'].isin(['dont_know', 'ambiguous'])  # was problem, now resolved
n_fixed = fixed.sum()
vol_fixed = jct_out.loc[fixed, 'volume'].sum()
n_remain  = still_resid.sum()
vol_remain = jct_out.loc[still_resid, 'volume'].sum()

print(f'Newly resolved by rules: {n_fixed:,} rows  ({vol_fixed/1e6:,.0f} Mgal)')
print(f'Still unresolved:        {n_remain:,} rows  ({vol_remain/1e6:,.0f} Mgal)')

print('\nUpdated source_type distribution:')
orig = jct['source_type'].value_counts().rename('original')
new  = jct_out['source_type'].value_counts().rename('final')
comp = pd.concat([orig, new], axis=1).fillna(0).astype(int)
comp['delta'] = comp['final'] - comp['original']
print(comp.sort_values('original', ascending=False).to_string())

=== Rule-fix results ===
Newly resolved by rules: 5,884 rows  (14,024 Mgal)
Still unresolved:        1,748 rows  (998 Mgal)

Updated source_type distribution:
                 original  final  delta
source_type                            
surface_direct      15876  17252   1376
no_source            8551   8551      0
impoundment          8013   8904    891
interconnection      5193   7166   1973
ambiguous            4678   1711  -2967
dont_know            2954     37  -2917
groundwater          2053   2869    816
reuse                1570   2384    814
srbc_only             475    489     14


In [20]:
# Overwrite the final parquet with rule-fixed types included
jct_out.to_parquet(UPDATED_PATH, index=False)
print(f'Saved updated junction table to {UPDATED_PATH}')
print(f'Columns: {jct_out.columns.tolist()}')

# Final volume coverage summary
print('\nVolume by final source_type:')
total_vol = jct_out['volume'].sum()
for stype, g in jct_out.groupby('source_type'):
    v = g['volume'].sum()
    print(f"  {stype:20s}: {v/total_vol*100:5.1f}%  ({v/1e6:>8,.0f} Mgal)")

Saved updated junction table to data/junction_dep_updated.parquet
Columns: ['api10', 'operator', 'planSource', 'managementPlanId', 'volume', 'saved_fn', 'operator_clean', 'site_clean', 'site_ID', 'source_type', 'search_name', 'nhd_id', 'nhd_name', 'nhd_ftype', 'nhd_layer', 'score', 'dist_km', 'match_tier', 'dep_score', 'dep_type', 'dep_lat', 'dep_lon', 'dep_src', 'dep_stype']

Volume by final source_type:
  ambiguous           :   0.7%  (     738 Mgal)
  dont_know           :   0.3%  (     260 Mgal)
  groundwater         :   2.5%  (   2,531 Mgal)
  impoundment         :  17.4%  (  17,356 Mgal)
  interconnection     :  21.8%  (  21,712 Mgal)
  no_source           :   0.1%  (     103 Mgal)
  reuse               :   2.8%  (   2,808 Mgal)
  srbc_only           :   0.4%  (     370 Mgal)
  surface_direct      :  54.0%  (  53,772 Mgal)


## 12. Manual curation

Export all remaining `dont_know` / `ambiguous` sources to `data/manual_curation.csv` for
hand-labelling in Excel. The CSV includes context columns to aid decisions.

**Workflow:**
1. Run this cell once to generate the CSV (skips if file already exists to protect edits)
2. Open `data/manual_curation.csv` in Excel
3. Fill in `curated_type` for entries you can resolve — valid values:
   `surface_direct`, `interconnection`, `impoundment`, `groundwater`, `reuse`, `srbc_only`
4. Leave `curated_type` blank for entries you cannot determine
5. Run the **Apply curation** cell below to fold your edits into the junction table

In [21]:
import os

CURATION_PATH = 'data/manual_curation.csv'
VALID_TYPES = {'surface_direct', 'interconnection', 'impoundment',
               'groundwater', 'reuse', 'srbc_only'}

if os.path.exists(CURATION_PATH):
    print(f'Curation file already exists — skipping export to protect existing edits.')
    print(f'Delete {CURATION_PATH} and re-run this cell to regenerate from scratch.')
else:
    # Gather residual rows after all automated steps
    resid_mask = jct_out['source_type'].isin(['dont_know', 'ambiguous'])

    # One row per unique planSource, with context to aid manual labelling
    resid_sources = (
        jct_out[resid_mask]
        .groupby('planSource')
        .agg(
            source_type   = ('source_type', 'first'),
            operator_clean= ('operator_clean', lambda x: x.mode().iloc[0] if len(x) else ''),
            volume_Mgal   = ('volume', lambda x: round(x.sum() / 1e6, 2)),
            dep_score     = ('dep_score', 'first'),
            dep_src       = ('dep_src', 'first'),
            dep_type      = ('dep_type', 'first'),
        )
        .reset_index()
        .sort_values('volume_Mgal', ascending=False)
    )

    resid_sources['curated_type'] = ''   # column to fill in Excel
    resid_sources['notes'] = ''          # optional free-text

    cols = ['planSource', 'curated_type', 'notes', 'source_type',
            'volume_Mgal', 'operator_clean', 'dep_score', 'dep_src', 'dep_type']
    resid_sources[cols].to_csv(CURATION_PATH, index=False)

    print(f'Exported {len(resid_sources):,} sources to {CURATION_PATH}')
    print(f'\nTop 20 by volume:')
    for _, r in resid_sources.head(20).iterrows():
        print(f"  {r['volume_Mgal']:>7.1f} Mgal  [{r['source_type']:9s}]  '{r['planSource'][:50]}'  "
              f"dep={r['dep_score']:.0f} -> '{str(r['dep_src'])[:35]}'")

Curation file already exists — skipping export to protect existing edits.
Delete data/manual_curation.csv and re-run this cell to regenerate from scratch.


### Apply curation

Run this cell after filling in `curated_type` values in the CSV.
It re-reads the file, validates entries, applies overrides to `jct_out`, and re-saves the parquet.
Safe to re-run as many times as needed as you refine the CSV.

In [22]:
if not os.path.exists(CURATION_PATH):
    print('No curation file found — run the export cell first.')
else:
    curation = pd.read_csv(CURATION_PATH)

    # Coerce blanks to empty string; map DEP term to our vocabulary
    curation['curated_type'] = (
        curation['curated_type'].fillna('').astype(str).str.strip()
        .str.replace(r'^surface_water$', 'surface_direct', regex=True)
    )
    filled = curation[curation['curated_type'] != '']

    bad = filled[~filled['curated_type'].isin(VALID_TYPES)]
    if len(bad):
        print(f'WARNING: {len(bad)} unrecognised curated_type values — fix before applying:')
        for _, r in bad.iterrows():
            print(f"  '{r['planSource']}' -> '{r['curated_type']}'")
    else:
        curation_lookup = filled.set_index('planSource')['curated_type']

        # Re-load the rule-fixed junction table so this cell is idempotent
        jct_apply = pd.read_parquet(UPDATED_PATH)

        applied = jct_apply['planSource'].isin(curation_lookup.index)
        jct_apply.loc[applied, 'source_type'] = jct_apply.loc[applied, 'planSource'].map(curation_lookup)

        jct_apply.to_parquet(UPDATED_PATH, index=False)

        n_applied = applied.sum()
        vol_applied = jct_apply.loc[applied, 'volume'].sum()
        still = jct_apply['source_type'].isin(['dont_know', 'ambiguous']).sum()

        print(f'Curation applied: {len(curation_lookup)} unique sources '
              f'-> {n_applied:,} junction rows  ({vol_applied/1e6:,.0f} Mgal)')
        print(f'Still unresolved (dont_know + ambiguous): {still:,} rows')

        print(f'\nFinal source_type distribution:')
        total = jct_apply['volume'].sum()
        for stype, g in jct_apply.groupby('source_type'):
            v = g['volume'].sum()
            print(f'  {stype:20s}: {v/total*100:5.1f}%  ({v/1e6:>8,.0f} Mgal)')

Curation applied: 231 unique sources -> 1,268 junction rows  (845 Mgal)
Still unresolved (dont_know + ambiguous): 480 rows

Final source_type distribution:
  ambiguous           :   0.2%  (     154 Mgal)
  groundwater         :   2.5%  (   2,531 Mgal)
  impoundment         :  17.5%  (  17,441 Mgal)
  interconnection     :  21.8%  (  21,771 Mgal)
  no_source           :   0.1%  (     103 Mgal)
  reuse               :   3.1%  (   3,093 Mgal)
  srbc_only           :   0.4%  (     370 Mgal)
  surface_direct      :  54.4%  (  54,187 Mgal)
